# **Knowledge Graph**

## **Imports**

In [ ]:
!pip install -q diffusers transformers accelerate "bitsandbytes>=0.46.1" matplotlib pandas wandb huggingface_hub pycocotools networkx Pillow safetensors torch torch_geometric torchvision tqdm gdown pyvis
import gc, gdown, IPython, json, os, random, shutil, torch, urllib.request, wandb, warnings, zipfile
import networkx as nx
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T

from diffusers import StableDiffusion3Pipeline
from huggingface_hub import notebook_login
from matplotlib import pyplot as plt
from PIL import Image
from pyvis.network import Network
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from typing import Counter

warnings.filterwarnings("ignore")

In [ ]:
COCO_ROOT = "/content/coco"
PSG_ROOT  = "/content/psg"

BATCH_SIZE = 4
IMG_SIZE = 512
LEARNING_RATE = 2e-6
LOG_WANDB_STEPS = 200
SEED = 42
STEPS = 2000
TRAIN_INFERENCE = 20

SELECTED_FUSION = "gnn_embeddings" # TODO changer par "cnn_embeddings", "gnn_embeddings" ou "urban_fusion" en fonction de ce que vous souhaitez exécuter

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## **Check GPU**

PyTorch peut utiliser CUDA pour exécuter les calculs sur un GPU NVIDIA. On vérifie avec torch.cuda.is_available() si CUDA est disponible et on choisit sinon le CPU. Il faut ensuite placer le modèle et les tenseurs sur le même device.

TensorFlow permet également d'utiliser un GPU. Utiliser TensorFlow et PyTorch dans un même projet est techniquement possible, mais généralement inutile puisqu'ils possèdent chacun leur propre système de tenseurs, de modèles et de calcul des gradients.

In [ ]:
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
    torch.cuda.manual_seed_all(SEED)
else:
    warnings.warn("⚠ No GPU — go to Runtime → Change runtime type → GPU", RuntimeWarning)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## **Login to Hugging face & W&B**

Hugging Face est une plateforme où l'on récupère et éventuellement publie des modèles/datasets. Se connecter est obligatoire pour ensuite télécharger Stable Diffusion.

Weights & Biases sert quant à lui au suivi des expériences d'entraînement : il permet d'enregistrer les métriques, les paramètres et les résultats afin de visualiser l'évolution de l'entraînement et comparer plusieurs essais.

In [ ]:
# --- Hugging Face ---
print("--- Hugging Face Login ---")
print("Get your token from: https://huggingface.co/settings/tokens")
notebook_login()

# --- Weights & Biases ---
print("\n--- Weights & Biases Login ---")
print("Get your API key from: https://wandb.ai/authorize")
wandb.login()

## **Dataset importation**

- `urllib.request.urlretrieve(...)` : télécharger une ressource accessible via une URL et à l’enregistrer localement

- `gdown.download_folder(...)` : bibliothèque conçue pour télécharger facilement des fichiers depuis Google Drive

In [ ]:
PSG_FOLDER_ID = "1VSRw_nnThLqvpmePCtaWnX_1Jmfg1Y1f"

COCO_URLS = {
    "panoptic": "http://images.cocodataset.org/annotations/panoptic_annotations_trainval2017.zip",
    "train2017": "http://images.cocodataset.org/zips/train2017.zip",
    "val2017":   "http://images.cocodataset.org/zips/val2017.zip",
    "annotations": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
}

PANOPTIC_NAME = ["panoptic_train2017.zip", "panoptic_val2017.zip"]

In [ ]:
def is_file_existing(path):
    if os.path.isdir(path) and os.listdir(path):
        print(f"✓ {path} already in /content — skipping")
        return True
    return False

def extract_zip(zip_path):
    print(f"\t└─ Extracting {zip_path} …")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(COCO_ROOT)
    os.remove(zip_path)

In [ ]:
if not is_file_existing(COCO_ROOT):
    os.makedirs(COCO_ROOT, exist_ok=True)

    for name, url in COCO_URLS.items():
        zip_path = f"/content/{name}.zip"

        print(f"Downloading {name} from official COCO servers …")
        urllib.request.urlretrieve(url, zip_path)
        extract_zip(zip_path)

        if name == "panoptic":
            panoptic_dir = os.path.join(COCO_ROOT, "annotations")

            for inner_zip in PANOPTIC_NAME:
                inner_zip_path = os.path.join(panoptic_dir, inner_zip)
                extract_zip(inner_zip_path)

        print(f"✓ {name} ready")

    print("✓ COCO ready at /content/coco")

In [ ]:
if not is_file_existing(PSG_ROOT):
    print("Downloading PSG folder …")
    gdown.download_folder(id=PSG_FOLDER_ID, output=PSG_ROOT, quiet=False, use_cookies=False)
    print("✓ PSG ready at /content/psg")

## **Data exploration**

In [ ]:
with open(f"{PSG_ROOT}/psg.json") as f:
    psg = json.load(f)

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(f"Data image: {len(psg['data']):,} images")

stuff_classes = psg['stuff_classes']
print(f"Stuff categories: {len(stuff_classes)}")

thing_classes = psg['thing_classes']
print(f"Thing categories: {len(thing_classes)}")

predicate_classes = psg['predicate_classes']
print(f"Predicate categories (relations): {len(predicate_classes)}")

object_classes = thing_classes + stuff_classes
print(f"Total categories: {len(object_classes)}")

In [ ]:
def make_hist(first, second, third, classes, type):
    counter = Counter()

    for item in psg[first]:
        for i in item[second]:
            counter[classes[i[third]]] += 1

    print(counter)

    top10 = counter.most_common(10)

    _, ax = plt.subplots(figsize=(10, 5))
    ax.barh([x[0] for x in top10][::-1], [x[1] for x in top10][::-1], color='pink')

    ax.set_xlabel(f'Occurence de {type}')
    ax.set_ylabel(f'{type}')
    ax.set_title(f'Top 10 {type}')

    plt.tight_layout()
    plt.show()

### **Relations les plus utilisées**

In [ ]:
make_hist('data', 'relations', 2, predicate_classes, 'relations')

### **Choses / objets les plus utilisés**

In [ ]:
make_hist('data', 'segments_info', 'category_id', object_classes, 'objects')

### **Affichage de psg['data']**

In [ ]:
pd.DataFrame(psg['data']).head(10)

### **Nombre de relations par image (moyenne)**

In [ ]:
relation_counts = [len(item['relations']) for item in psg['data']]
mean_relations = sum(relation_counts) / len(relation_counts)
print(f"Mean relations per image: {mean_relations:.2f}")

### **Nombre d'objets par image (moyenne)**

In [ ]:
counter_smthg = 0

for item in psg['data']:
    counter_smthg += len(item['segments_info'])

mean_smthg = counter_smthg / len(psg['data'])
print(f"Mean objects per image: {mean_smthg:.2f}")

## **Image examples**

In [ ]:
def show_image(img, idx):
    _, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(img)
    ax.set_title(f"Image: #{idx}", fontsize=11)
    ax.axis('off')

    plt.show()

In [ ]:
def show_example(idx=None):
    if idx is None:
        idx = random.randrange(len(psg['data']))

    item = psg['data'][idx]
    img_path = os.path.join(COCO_ROOT, item['file_name'])
    img = Image.open(img_path)

    segments = item['segments_info']
    obj_names = [object_classes[seg['category_id']] for seg in segments]

    triplets = []
    for s, o, p in item['relations']:
        triplets.append(f"{obj_names[s]} -[{predicate_classes[p]}] → {obj_names[o]}")

    show_image(img, idx)

    print("Objects:", obj_names)
    print ("Relations:")
    for t in triplets:
        print(f" • {t}")

    return item, img, img.size, obj_names, triplets

In [ ]:
sample_item, image, img_size, sample_objs, sample_triplets = show_example()

## **Knowledge graph representation**

Un Knowledge Graph représente une scène sous forme de triplets : `sujet - prédicat - objet`. Les sujets et objets correspondent aux noeuds du graphe, tandis que les prédicats représentent les relations / arêtes entre ces noeuds.

La Knowledge Graph Representation consiste ici à transformer les informations structurées d'un graphe de connaissances en une représentation textuelle.

### **Triplets**
- A rural scene with peson, car, road. A person is Walking on road. car is parked on road. car is beside building
- A rural scene with  [obj1] [obj2] [obj3] [obj4] .. [objN]. [obj_a] is [predicate] [obj_b]. [obj_c] is [predicate] [obj_d] ….

In [ ]:
def scene_graph_to_prompt(item, classes, predicate_classes, max_tokens=77):
    segments = item['segments_info']
    obj_names = [classes[seg['category_id']] for seg in segments]
    unique_objs = list(dict.fromkeys(obj_names))

    header = "A scene with " + ", ".join(unique_objs) + "."

    clauses = []
    for s, o, p in item['relations']:
        subj = obj_names[s]
        obj = obj_names[o]
        pred = predicate_classes[p]
        clauses.append(f"{subj} is {pred} {obj}")

    prompt = header
    for c in clauses:
        candidate = prompt + " " + c + "."
        if len(candidate) // 4 > max_tokens:
            break
        prompt = candidate

    return prompt


def scene_graph_to_baseline_prompt(item, classes):
    segments = item['segments_info']
    obj_names = [classes[seg['category_id']] for seg in segments]
    unique_objs = list(dict.fromkeys(obj_names))
    return "A scene with " + ", ".join(unique_objs) + "."

In [ ]:
sample_item, image, img_size, sample_objs, sample_triplets = show_example()
print("\n")

print("=" * 60)
print("Prompt amélioré:")
print(scene_graph_to_prompt(sample_item, object_classes, predicate_classes))
print("=" * 60)
print("Prompt de base:")
print(scene_graph_to_baseline_prompt(sample_item, object_classes))
print("=" * 60)

## **Urban scene filtering**

Tout d'abord on définit une liste d'objets correspondants au thème urbain. Puis, pour chaque image, on regarde les objets détectés ou présents dans sa représentation structurée et on vérifie si au moins l’un d’entre eux appartient à cette liste.

In [ ]:
URBAN_CATEGORY_NAMES = {
    'car', 'bus', 'truck', 'motorcycle', 'bicycle', 'person',
    'traffic light', 'stop sign', 'parking meter',
    'road', 'building-other-merged', 'bridge', 'pavement-merged',
    'fence-merged', 'tree-merged', 'sky-other-merged'
}

URBAN_IDS = set()
for i, c in enumerate(object_classes):
    if c in URBAN_CATEGORY_NAMES:
        URBAN_IDS.add(i)
print(f"Urban category IDs: {sorted(URBAN_IDS)}")
print(f"Categories: {[object_classes[i] for i in sorted(URBAN_IDS)]}")

urban_data = []
for item in psg['data']:
    cats = set(seg['category_id'] for seg in item['segments_info'])
    has_vehicle_or_road = bool(cats & {2, 3, 5, 7, 100, 123})
    has_structure = bool(cats & {82, 129, 117, 116, 119})
    if has_vehicle_or_road and has_structure:
        urban_data.append(item)

print(f"\n Urban scenes selected: {len(urban_data):,} / {len(psg['data']):,} "
f"({len(urban_data)/len(psg['data'])*100:.1f}%)")

## **Visualizing predicates with graph**

`DiGraph` : graphe orienté, essentiel ici pour illustrer la relation sujet → objet

Pour chaque triplet, on crée une arête (edge) allant du sujet vers l’objet (qui sont des nodes), et le prédicat devient le label de cette relation.

In [ ]:
sample_item, image, img_size, sample_objs, sample_triplets = show_example()

G = nx.DiGraph()

segments = sample_item['segments_info']
obj_names = [object_classes[seg['category_id']] for seg in segments]

for s, o, p in sample_item['relations']:
    subj_label = f"{obj_names[s]} (ID:{s})"
    obj_label = f"{obj_names[o]} (ID:{o})"
    pred = predicate_classes[p]

    G.add_node(s, label=subj_label, title=f"Class: {obj_names[s]}", color="#D5A6BD", shape="box")
    G.add_node(o, label=obj_label, title=f"Class: {obj_names[o]}", color="#B4A7D6", shape="box")
    G.add_edge(s, o, label=pred, color="#696969", width=2)

net = Network(notebook=True, directed=True, height="500px", width="100%", cdn_resources='in_line')
net.from_nx(G)

net.set_options("""
    var options = {
        "nodes": {
            "font": { "size": 18, "face": "Tahoma" },
            "borderWidth": 2,
            "shadow": true
        },
        "edges": {
            "arrows": { "to": { "enabled": true, "scaleFactor": 1.2 } },
            "font": { "size": 15, "align": "middle", "face": "Tahoma", "color": "#333333"},
            "smooth": { "type": "continuous", "forceDirection": "none" }
        },
        "physics": {
            "barnesHut": {
                "gravitationalConstant": -3000,
                "centralGravity": 0.3,
                "springLength": 150
                },
            "minVelocity": 0.75,
            "stabilization": {
                "enabled": true,
                "iterations": 1000
            }
        }
    }
""")

net.show("scene_graph.html")
IPython.display.HTML("scene_graph.html")

## **ControlNet**

ControlNet utilise une représentation segmentée de l'image dans laquelle les différentes catégories d'objets sont représentées par des couleurs spécifiques. Cette segmentation fournit au modèle une information structurelle sur la position, la forme et l'espace occupé par les différents éléments. ControlNet conditionne ensuite Stable Diffusion afin que l'image générée respecte davantage cette structure.

In [ ]:
def generate_segmentation_map(item, img_width, img_height, all_classes, coco_root):
    file_name = item['file_name']
    base_name = os.path.basename(file_name)

    split = "train" if "train" in file_name else "val"
    panoptic_img_name = base_name.replace('.jpg', '.png')
    panoptic_path = os.path.join(coco_root, f'panoptic_{split}2017', panoptic_img_name)

    if not os.path.exists(panoptic_path):
        print(f"Mask introuvable : {panoptic_path}")
        return Image.new('RGB', (img_width, img_height), (0, 0, 0))

    panoptic_img = Image.open(panoptic_path)

    # Charger l'image Panoptic (chaque pixel contient l'ID de l'objet)
    pan_im = np.array(panoptic_img.convert("RGB"), dtype=np.uint32)
    seg_map = pan_im[:, :, 0] + pan_im[:, :, 1] * 256 + pan_im[:, :, 2] * (256**2)

    # Créer une image vide où l'on va "peindre" chaque segment
    color_map = np.zeros((img_height, img_width, 3), dtype=np.uint8)

    # Générer un dictionnaire de couleurs uniques par catégorie
    np.random.seed(SEED)
    colors = np.random.randint(30, 255, size=(len(all_classes), 3), dtype=np.uint8)

    # Remplir l'image avec les couleurs correspondantes au pixel près !
    for seg in item['segments_info']:
        cat_id = seg['category_id']
        s_id = seg['id']

        # Trouver tous les pixels appartenant à cet objet
        mask = (seg_map == s_id)

        # Appliquer la couleur de SA classe sur ces pixels
        color_map[mask] = colors[cat_id]

    return Image.fromarray(color_map).resize((img_width, img_height), Image.NEAREST)

In [ ]:
# Génération et affichage
sample_item, image, img_size, sample_objs, sample_triplets = show_example()
img_width, img_height = img_size
seg_img = generate_segmentation_map(sample_item, img_width, img_height, object_classes, COCO_ROOT)

fig, axs = plt.subplots(1, 2, figsize=(15, 7))
axs[0].imshow(image)
axs[0].set_title("Original Image", fontsize=14)
axs[0].axis('off')

axs[1].imshow(seg_img)
axs[1].set_title("ControlNet : Pixel-Perfect Segmentation Map", fontsize=14)
axs[1].axis('off')

plt.tight_layout()
plt.show()

## **GNN**

Le GNN prend un graphe comme entrée et produit un embedding numérique qui résume la structure de ce graphe. Le résultat produit est de la forme : `tensor([-0.0867, 0.0344, -0.0290, ...])`. C'est un vecteur de caractéristiques, ou embedding.

### **build_scene_graph_data**

On transforme la scène en un graphe numérique dans lequel les nœuds correspondent aux objets et sont initialement représentés par leur ID de catégorie. edge_index contient les connexions orientées entre ces objets. Le type de relation (holding, next to, etc.) n'est pas utilisé par ce GNN.

In [ ]:
def build_scene_graph_data(item):
    segments = item['segments_info']
    node_classes = [seg['category_id'] for seg in segments] # récupération des identifiants numériques de catégories ; ex : node_classes = [1, 3, 7]
    num_nodes = len(node_classes)

    if num_nodes == 0:
        raise ValueError("Image has no object segments.")

    x = torch.tensor(node_classes, dtype=torch.long).unsqueeze(1) # les éléments de node_classes deviennent des noeuds ; ex : x = [[1], [3], [7]]

    edges = []
    for source, target, _ in item['relations']:
        edges.append([source, target]) # on ajoute les relations sans leur nom, juste leur sens (de A vers B)

    """
    PyTorch Geometric utilise ce format :
        - première ligne  = sources
        - deuxième ligne = destinations
    ex : avant [[0, 1], [0, 2]], après [[0, 0], [1, 2]]
    """
    if len(edges) > 0:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    return Data(x=x, edge_index=edge_index)

### **SceneGraphGNN**

Le paramètre `embedding_dim=256` signifie que chaque catégorie d'objet (ici des identifiants) est représenté par un vecteur de 256 nombres.

Les `GCNConv` permettent aux noeuds de mettre à jour leur représentation en utilisant les informations provenant des noeuds auxquels ils sont connectés. Chaque objet n'est donc progressivement plus décrit uniquement comme : *« je suis une personne »* mais davantage comme : *« je suis une personne située dans une structure de graphe où je suis connectée à une voiture et un chien »*.

Après les GCN chaque objet a toujours son propre vecteur hors ce qui est attendu c'est un seul embedding représentant toute l'image / toute la scène. Pour cela, on utilise `global_mean_pool` qui fait une moyenne des représentations des noeuds appartenant au même graphe. Ainsi :
<small>
```python
20 embeddings de 512 dimensions
              ↓
        global_mean_pool
              ↓
1 embedding de 512 dimensions
```
</small>

Enfin, `readout` (MLP) permet de modifier la dimension de l'embedding du graphe de 512 à 1152. Ceci explique la sortie *sample_embedding.shape=torch.Size([1152])*.

In [ ]:
class SceneGraphGNN(nn.Module):

    def __init__(self, num_obj_classes, embedding_dim=256, hidden_dim=512, output_dim=1152):
        super().__init__()
        self.obj_embedding = nn.Embedding(num_obj_classes, embedding_dim) # construit un embedding (num_obj_classes vecteurs chacun de 256)

        self.conv1 = GCNConv(embedding_dim, hidden_dim) # information du voisinage proche
        self.conv2 = GCNConv(hidden_dim, hidden_dim) # information propagée plus loin
        self.conv3 = GCNConv(hidden_dim, hidden_dim) # représentation encore plus contextuelle

        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, data):
        x = self.obj_embedding(data.x.squeeze(1))
        x = F.relu(self.conv1(x, data.edge_index))
        x = F.relu(self.conv2(x, data.edge_index))
        x = F.relu(self.conv3(x, data.edge_index))

        graph_emb = global_mean_pool(x, data.batch)
        return self.readout(graph_emb)

### **get_image_emmebding**

PyTorch Geometric peut traiter plusieurs graphes simultanément. Il a donc besoin de savoir : **« À quel graphe appartient chaque nœud ? »**. Pour cela, on fait :
<small>
```python
data.batch = torch.zeros(
    data.num_nodes,
    dtype=torch.long,
    device=device
)
```
</small>
qui permet d'associer chaque noeud au même graphe (ici 0).

`model.eval()` : permet de mettre le modèle en mode évaluation / inférence. On indique que l'on souhaite utiliser le modèle pour produire un résultat, pas l'entraîner.

`torch.no_grad()` : pendant un entraînement, PyTorch doit mémoriser énormément d'informations afin de calculer les gradients. Puisqu'on ne veut pas entraîner le modèle, nous n'avons pas besoin des gradients c'est pourquoi on désactive l'utilisation des gradients pour économiser de la mémoire, des calculs et les ressources GPU.

Enfin, on fait passer nos données dans le modèle.

In [ ]:
def get_image_embedding(item, model, device=DEVICE):
    data = build_scene_graph_data(item)
    data = data.to(device)
    data.batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
    model.eval()
    with torch.no_grad():
        emb = model(data)
    return emb.squeeze(0).cpu()

In [ ]:
model = SceneGraphGNN(num_obj_classes=len(object_classes), embedding_dim=256, hidden_dim=512, output_dim=1152).to(DEVICE)

sample_embedding = get_image_embedding(sample_item, model, device=DEVICE)
print("Sample image scene-graph embedding shape:", sample_embedding.shape)
print(f"  Embedding preview (first 10 dims): {sample_embedding[:10]}")

### **Résumé GNN**

- création du modèle :

  - nn.Embedding crée une table contenant num_obj_classes embeddings de 256 dimensions ; ces vecteurs sont initialement appris initialisés comme paramètres du modèle ;

  - trois GCNConv vont permettre aux nœuds d’échanger de l’information en suivant les connexions indiquées dans edge_index ;

  - le MLP final transforme la représentation globale de 512 dimensions en un embedding de 1152 dimensions ;

- création d’un objet Data :

  - x = [[1], [3], [7]] contient les IDs de catégories des nœuds ;

  - edge_index = [[sources], [destinations]] décrit les connexions orientées ;

- data.batch = torch.zeros(...) indique que tous les nœuds appartiennent ici au même graphe ;

- model.eval() place le modèle en mode inférence ;

- torch.no_grad() désactive le calcul des gradients, ce qui réduit mémoire et calculs ;

- les données passent dans le modèle :

  - les IDs sont transformés en embeddings de 256 dimensions ;

  - les trois couches GCN mettent à jour chaque nœud à partir de son voisinage ;

  - global_mean_pool agrège tous les nœuds pour produire un vecteur de 512 dimensions représentant le graphe entier ;

  - le MLP transforme enfin ce vecteur de 512 → 1152.

## **ControlNet and GNN merging**

Dans le code fourni, le GNN, le CNN de segmentation et le bloc de fusion sont tous créés puis directement utilisés en inférence sans entraînement préalable visible.

### **ControlNetSegmentationEncoder**

- Encode l'image de segmentation générée par le CNN

- Les couches de convolutions permettent de produire une représentation de la structure spatiale de la segmentation. MaxPool2d permet de réduire la résolution spaciale et ainsi de diminuer le coût de calcul, d'augmenter le champ de vision du réseau et de conserver les caractéristiques importantes

- `self.projection` : après le CNN, le résultat obtenu ressemble à [1, 256, hauteur, largeur] *(forme : [batch, channels, hauteur, largeur])*

  - `nn.AdaptiveAvgPool2d((8, 8))` force la sortie spaciale à avoir une taille de 256 x 8 x 8, donc on a [1, 256, 8, 8]

  - `nn.Flatten(1)` transforme 256 x 8 x 8 en 16384 *(= 256\*8\*8)* - donc on a [1, 16384]

  - `nn.Linear(hidden_dims[3] * 8 * 8, output_dim)` transforme 16384 en 1152

  - `nn.LayerNorm(output_dim)` normalise les activations de l’embedding pour obtenir des représentations numériquement plus stables

In [ ]:
class ControlNetSegmentationEncoder(nn.Module):
    def __init__(self, in_channels=3, hidden_dims=(32, 64, 128, 256), output_dim=1152):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dims[0], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(hidden_dims[0], hidden_dims[1], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(hidden_dims[1], hidden_dims[2], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(hidden_dims[2], hidden_dims[3], kernel_size=3, padding=1),
            nn.ReLU(),
        )

        self.projection = nn.Sequential(
            nn.AdaptiveAvgPool2d((8, 8)),
            nn.Flatten(1),
            nn.Linear(hidden_dims[3] * 8 * 8, output_dim),
            nn.LayerNorm(output_dim),
        )

    def forward(self, segmentation_tensor):
        features = self.encoder(segmentation_tensor)
        return self.projection(features)

### **SceneFusionBlock**

`self.net` permet au réseau de combiner les informations du GNN et du CNN. C'est en réalité un petit MLP dont le but est d'apprendre comment combiner l’information structurelle du graphe et l’information spatiale de la segmentation

In [ ]:
class SceneFusionBlock(nn.Module):
    def __init__(self, embedding_dim=1152, hidden_dim=1536):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim),
            nn.LayerNorm(embedding_dim),
        )

    def forward(self, embedding, controlnet_embedding):
        fused_input = torch.cat([embedding, controlnet_embedding], dim=1)
        return self.net(fused_input)

In [ ]:
seg_encoder = ControlNetSegmentationEncoder().to(DEVICE)

image_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)), # mettre toutes les images à la même taille
    T.ToTensor(), # convertit l’image en tenseur PyTorch
    T.Normalize([0.5], [0.5])
])

In [ ]:
fusion_block = SceneFusionBlock().to(DEVICE)

cnn_embeddings = []
gnn_embeddings = []
fused_results = []

### **Boucle sur les images urbaines**

- Création de la segmentation (CNN) avec *seg_img = generate_segmentation_map(...)* donc image → segmentation map

- Production de l'embedding GNN avec *gnn_emb = get_image_embedding(...)* donc image → embedding

- Préparation de la segmentation avec *seg_tensor = image_transform(seg_img)*, *.unsqueeze(0)* qui ajoute la dimension batch ([C, H, W] → [1, C, H, W]) et enfin *.to(DEVICE)* qui envoie l'entrée sur le GPU si CUDA est utilisé

- Encodage de la segmentation avec *controlnet_emb = seg_encoder(seg_tensor)* donc segmentation → embedding [1, 1152]

- Transformation du GNN avec *.unsqueeze(0)* pour ajouter le même format batch que controlnet embedding [1, 1152]. Ceci est essentiel pour pouvoir les concaténer

- Lors de la fusion, gnn_emb 1152 + cnn_emb 1152 → résultat 2304. Grâce au `fusion_block` on passe de 2304 à 1152

Dans le code fourni, le GNN, le CNN de segmentation et le bloc de fusion sont tous créés puis directement utilisés en inférence sans entraînement préalable visible. Leurs poids restent donc initialisés aléatoirement. La pipeline est architecturalement cohérente, mais les représentations produites n’ont pas appris à capturer la sémantique attendue.

In [ ]:
with torch.no_grad(): # Désactive le suivi des opérations. Ne construis pas le graphe de calcul.
    for image in urban_data:
        img_width = image['width']
        img_height = image['height']
        seg_img = generate_segmentation_map(image, img_width, img_height, object_classes, COCO_ROOT) # ControlNet
        gnn_emb = get_image_embedding(image, model, device=DEVICE) # GNN

        seg_tensor = image_transform(seg_img).unsqueeze(0).to(DEVICE)
        controlnet_emb = seg_encoder(seg_tensor)

        if not isinstance(gnn_emb, torch.Tensor):
            gnn_emb = torch.tensor(gnn_emb).to(DEVICE)

        gnn_emb = gnn_emb.to(DEVICE)

        if gnn_emb.dim() == 1:
            gnn_emb = gnn_emb.unsqueeze(0)

        cnn_embeddings.append(controlnet_emb.cpu())
        gnn_embeddings.append(gnn_emb.cpu())
        final_fusion = fusion_block(gnn_emb, controlnet_emb)

        fused_results.append(final_fusion.detach().cpu())

In [ ]:
cnn_embeddings = torch.cat(cnn_embeddings, dim=0)
torch.save(cnn_embeddings, "cnn_embeddings.pt")
print("Embeddings enregistrés !")

cnn_embeddings = torch.load("cnn_embeddings.pt", map_location="cpu")

print(cnn_embeddings.shape)
print(cnn_embeddings.dtype)
print(torch.isnan(cnn_embeddings).any())
print(torch.isinf(cnn_embeddings).any())
print(cnn_embeddings.mean(), cnn_embeddings.std(), cnn_embeddings.min(), cnn_embeddings.max())

In [ ]:
gnn_embeddings = torch.cat(gnn_embeddings, dim=0)
torch.save(gnn_embeddings, "gnn_embeddings.pt")
print("Embeddings enregistrés !")

gnn_embeddings = torch.load("gnn_embeddings.pt", map_location="cpu")

print(gnn_embeddings.shape)
print(gnn_embeddings.dtype)
print(torch.isnan(gnn_embeddings).any())
print(torch.isinf(gnn_embeddings).any())
print(gnn_embeddings.mean(), gnn_embeddings.std(), gnn_embeddings.min(), gnn_embeddings.max())

In [ ]:
all_fusions = torch.cat(fused_results, dim=0)
torch.save(all_fusions, "urban_fusion_embeddings.pt")
print("Embeddings enregistrés !")

urban_fusion = torch.load("urban_fusion_embeddings.pt", map_location="cpu")

print(urban_fusion.shape)
print(urban_fusion.dtype)
print(torch.isnan(urban_fusion).any())
print(torch.isinf(urban_fusion).any())
print(urban_fusion.mean(), urban_fusion.std(), urban_fusion.min(), urban_fusion.max())

## **Adapter Stable 3.5**

### **Chargement du modèle**

In [ ]:
# Libère la mémoire non utilisée
gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"

pipe = StableDiffusion3Pipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16, # Permet d'utiliser moins de mémoire GPU
    text_encoder_3=None, # Désactive une partie du modèle pour économiser de la mémoire
    tokenizer_3=None, # Désactive une partie du modèle pour économiser de la mémoire
    low_cpu_mem_usage=True, # Limite la consommation mémoire au chargement
)

pipe.enable_model_cpu_offload()

### **Conversion des prompts en embedding & test du modèle**

- Les `positive` prompts représentent ce que l'on souhaite garder sur notre image. Ici c'est une sorte de fusion des 1 et 2 convertit en embedding. C’est cette représentation détaillée qui indique au modèle ce qu’on veut générer.

- Les `negative` prompts à l'inverse représentent ce que l'on souhaite éviter.

- Contrairement au embedding normaux qui conversent une représentation token par token, les `pooled_embedding` sont une version globale et condensée du prompt.

In [ ]:
prompt_1 = "Highly detailed, 35mm photograph, sharp focus, architectural realism, cinematic lighting, realistic textures, f/2.8"
prompt_2 = "An urban street scene"
negative_prompt = "Anime, cartoon, 3D render, illustration, surrealism, distorted architecture, blurry, low quality, bad anatomy, deformed cars, over-saturated, fantasy elements"

with torch.no_grad():
    positive_embedding, negative_embedding, positive_pooled_embedding, negative_pooled_embedding = pipe.encode_prompt(
        prompt=prompt_1, # Pour CLIP L (Text Encoder 1)
        prompt_2=prompt_2, # Pour CLIP G (Text Encoder 2)
        prompt_3=None, # Pour T5-XXL (Text Encoder 3 - Désactivé pour économiser la VRAM)
        negative_prompt=negative_prompt,
        negative_prompt_2=negative_prompt,
        negative_prompt_3=None,
        device=DEVICE,
        num_images_per_prompt=1,
        do_classifier_free_guidance=True,
    )

print("positive_embedding:", positive_embedding.shape)
print("negative_embedding:", negative_embedding.shape)
print("positive_pooled_embedding:", positive_pooled_embedding.shape)
print("negative_pooled_embedding:", negative_pooled_embedding.shape)

#### **Test du modèle non entraîné**

L'inférence correspond à la phase de génération, au cours de laquelle le modèle entraîné ou pré-entraîné est utilisé sans mise à jour de ses poids.

In [ ]:
image = pipe(prompt_1, prompt_2, num_inference_steps=TRAIN_INFERENCE, guidance_scale=4.5).images[0]
image.save("BaselineImage.png")

image

### **Gel de Stable Diffusion 3.5**

On empêche Stable Diffusion de s'entraîner. Seul l'adapter sera mis à jour.

In [ ]:
pipe.vae.requires_grad_(False)
pipe.transformer.requires_grad_(False)

if pipe.text_encoder is not None:
    pipe.text_encoder.requires_grad_(False)

if pipe.text_encoder_2 is not None:
    pipe.text_encoder_2.requires_grad_(False)

if pipe.text_encoder_3 is not None:
    pipe.text_encoder_3.requires_grad_(False)

### **Entrainement du modèle sur le GNN, CNN et la fusion des deux**

#### **Définir l’adapter 8 tokens**

Cette classe sert à générer des tokens pour chaque image (x). Ceux-ci contiennent les informations provenant du CNN/GNN. Stable Diffusion recevra donc le prompt textuel et les tokens structurels.

Dans la fonction **train_adapter**, `loss = mse(model_pred, target)` permet de répondre à la question : "Les tokens produits par l’Adapter ont-ils aidé Stable Diffusion à faire la bonne prédiction ?"

In [ ]:
class LightweightUrbanFusionAdapter(nn.Module):
    def __init__(self, input_dim, num_tokens=8, hidden_dim=4096):
        super().__init__()

        # Nombre de tokens artificiels ajoutés au prompt
        self.num_tokens = num_tokens

        # Taille d'un token dans Stable Diffusion 3.5
        self.hidden_dim = hidden_dim

        # Petit réseau de neurones qui transforme le vecteur fusionné en tokens
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024), # Étape 1 : Réduction / Compression
            nn.SiLU(), # Étape 2 : Activation non-linéaire
            nn.LayerNorm(1024), # Étape 3 : Stabilisation
            nn.Linear(1024, num_tokens * hidden_dim), # Étape 4 : Expansion vers SD 3.5
        )

    def forward(self, x):
        x = self.net(x)
        return x.view(x.size(0), self.num_tokens, self.hidden_dim)

#### **Sauvegarder l’adapter initial - comme une seed**

Peut servir à :
- reproduire ou comparer l’état initial ;
- recommencer l’entraînement depuis exactement cette architecture/configuration ;
- comparer avant/après ;
- éviter de perdre l’initialisation de référence.

In [ ]:
def save_adapter(adapter, fusion, fusion_name):
    torch.save({
        "adapter_state_dict": adapter.state_dict(),
        "input_dim": fusion.shape[-1],
        "num_tokens": 8,
        "hidden_dim": 4096,
    }, f"{fusion_name}_adapter_init.pt")

    print("Adapter saved.")

#### **Préparation du dataset de fusion**

Cette classe permet de récupérer les informations d'une image spécifique. L'embedding représente ici le CNN ou le GNN ou la fusion des 2 selon la valeur de la variable SELECTED_FUSION.

In [ ]:
# Dataset personnalisé qui associe :
# - une image réelle
# - son ControlNet ou son GNN ou la fusion des deux
# - son chemin relatif
class UrbanFusionDataset(Dataset):
    def __init__(self, urban_data, fusion, coco_root, image_transform=None):
        self.urban_data = urban_data
        self.fusion = fusion
        self.coco_root = coco_root
        self.transform = image_transform

        assert len(self.urban_data) == len(self.fusion), \
            f"Mismatch: urban_data={len(self.urban_data)} / fusion={len(self.fusion)}"

    def __len__(self):
        # Nombre total d'exemples dans le dataset
        return len(self.fusion)

    def __getitem__(self, idx):
        # Récupération des informations de l'image
        item = self.urban_data[idx]

        image_path = os.path.join(self.coco_root, item["file_name"])
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        embedding = self.fusion[idx]

        return {
            "image": image,
            "embedding": embedding,
            "image_path": image_path,
        }

#### **Suivi d'entraînement avec WandB**

On utilise Weights & Biases pour suivre la loss et les paramètres pendant l'entraînement.

In [ ]:
def run_wandb(adapter, name):
    num_params = sum(p.numel() for p in adapter.parameters() if p.requires_grad)

    wandb.init(
        project="gnn-fusion-sd35",
        name=name,
        config={
            "input_dim": 1152,
            "num_tokens": 8,
            "hidden_dim": 4096,
            "adapter_params": num_params,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "prompt": prompt_1,
            "prompt_2": prompt_2
        }
    )

    wandb.watch(adapter, log="gradients", log_freq=50)

### **Entraînement de l'adapter**

In [ ]:
def get_batch_values(batch):
    image = batch["image"].to(DEVICE, dtype=torch.float16)
    embedding = batch["embedding"].to(DEVICE, dtype=torch.float32)

    return image, embedding

La fonction `flow_matching` construit le problème d’apprentissage :

- Le `latent` correspond à une représentation compressée de l'image réelle. Il contient encore l’information visuelle importante de l’image — formes, structures, textures, organisation générale — mais sous une forme beaucoup plus compacte et adaptée aux calculs de Stable Diffusion.

- `noise = torch.randn_like(latents)` génère du bruit aléatoire

- `timesteps = torch.randint(...)` choisit pour chaque image du batch un niveau aléatoire dans le processus de bruitage - il indique à quel niveau du processus de bruitage se trouve chaque latent

  - timestep faible → latent encore très proche de l'image

  - timestep moyen → image et bruit mélangés

  - timestep élevé → latent presque entièrement bruité

- `target = noise - latents` représente la direction de transformation entre les données réelles et le bruit. Le Transformer doit apprendre/prédire cette direction.

On ajoute volontairement du bruit aux latents des images afin de créer différents états intermédiaires entre une image réelle et du bruit pur. Le modèle doit ensuite prédire la direction permettant de revenir vers les données réelles. C'est cette capacité qui lui permet, lors de l'inférence, de partir d'un bruit aléatoire et de générer progressivement une image cohérente.

In [ ]:
def flow_matching(latents):
    num_train_timesteps = 1000

    noise = torch.randn_like(latents)
    batch_size = latents.shape[0]

    timesteps = torch.randint(0, num_train_timesteps, (batch_size,), device=latents.device).long()

    sigmas = timesteps.float() / num_train_timesteps
    sigmas = sigmas.view(-1, 1, 1, 1).to(dtype=latents.dtype)

    noisy_latents = (1.0 - sigmas) * latents + sigmas * noise
    target = noise - latents

    return noisy_latents, target, timesteps

In [ ]:
def wandb_log(avg_loss, loss, optimizer, alpha_train, grad_norm, global_steps, item_name, adapter, embedding):
    adapter.eval()

    with torch.no_grad():
        fusion_example = embedding[:1].to(DEVICE, dtype=torch.float32)
        fusion_tokens = adapter(fusion_example)
        fusion_tokens = torch.tanh(fusion_tokens) * alpha_train
        fusion_tokens = fusion_tokens.to(dtype=positive_embedding.dtype)

        conditioned_positive_embedding = torch.cat([positive_embedding, fusion_tokens], dim=1)
        conditioned_negative_embedding = torch.cat([negative_embedding, torch.zeros_like(fusion_tokens)], dim=1)

        image = pipe(
            prompt_embeds=conditioned_positive_embedding,
            negative_prompt_embeds=conditioned_negative_embedding,
            pooled_prompt_embeds=positive_pooled_embedding,
            negative_pooled_prompt_embeds=negative_pooled_embedding,
            num_inference_steps=TRAIN_INFERENCE,
            guidance_scale=4.0,
            height=IMG_SIZE,
            width=IMG_SIZE,
        ).images[0]

    adapter.train()

    wandb.log({
        "train/loss": loss.item(),
        f"train/avg_loss_{LOG_WANDB_STEPS}_steps": avg_loss,
        "train/lr": optimizer.param_groups[0]["lr"],
        "train/alpha_train": alpha_train,
        "train/grad_norm": float(grad_norm),
        "sample/test_image": wandb.Image(
            image,
            caption=f"{item_name} - step {global_steps}"
        ),
    }, step=global_steps)

#### **Boucle d'entraînement**

In [ ]:
def save_training_checkpoint(adapter, optimizer, steps, fusion, alpha_train, checkpoint_path):
    bad_params = [name for name, param in adapter.named_parameters() if not torch.isfinite(param).all()]
    if bad_params:
        raise RuntimeError(f"Checkpoint refuse: parametres non finis dans {bad_params}")

    torch.save({
        "adapter_state_dict": adapter.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "steps": steps,
        "input_dim": fusion.shape[-1],
        "num_tokens": 8,
        "hidden_dim": 4096,
        "prompt": prompt_1,
        "prompt_2": prompt_2,
        "alpha_train": alpha_train,
    }, checkpoint_path)

    wandb.save(checkpoint_path)
    print("Checkpoint saved:", checkpoint_path)

La fonction `train_adapter` prend en paramètre un DataLoader. Celui ci construit automatiquement des batchs lesquels contiennent chacun BATCH_SIZE images avec leur embedding associés.

`steps` correspond ici au nombre de batchs / mises à jour de l'optimizer que l'on souhaite effectuer.

Les negative prompts sont ignorés car le but est seulement d'entraîner l'adapter à fournir un bon conditionnement positif pour que le Transformer prédise correctement le flow.

Pendant chaque batch d’entraînement, Stable Diffusion ne génère pas une image complète. Il fait seulement une prédiction dans l’espace latent à un timestep donné.
<small>
```js
embedding CNN/GNN → Adapter → 8 tokens artificiels → prompt + tokens -- image réelle → VAE → latent réel → flow matching → latent bruité
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
↓
SD Transformer gelé → model_pred → MSE avec target → Loss → Adapter mis à jour
```
</small>

In [ ]:
def train_adapter(data_loader, steps, adapter, alpha_train, optimizer, item_name):
    progress_bar = tqdm(total=steps, desc=f"{steps} steps")
    global_steps = 0
    running_count = 0
    running_loss = 0

    print(f"{'Step':>6} {'Training Loss':>15} {'Avg Loss':>15} {'Grad Norm':>12}")

    while global_steps < steps:
        for batch in data_loader:
            if global_steps >= steps: break

            image, embedding = get_batch_values(batch)

            with torch.no_grad():
                latents = pipe.vae.encode(image).latent_dist.sample()
                latents = latents * pipe.vae.config.scaling_factor

            noisy_latents, target, timesteps = flow_matching(latents)

            with torch.no_grad():
                positive_embedding, _, positive_pooled_embedding, _ = pipe.encode_prompt(
                    prompt=[prompt_1] * image.shape[0],
                    prompt_2=[prompt_2] * image.shape[0],
                    prompt_3=None,
                    negative_prompt=negative_prompt,
                    negative_prompt_2=negative_prompt,
                    negative_prompt_3=None,
                    device=DEVICE,
                    num_images_per_prompt=1,
                    do_classifier_free_guidance=False,
                )

            # Fusion tokens
            fusion_tokens = adapter(embedding.float())

            # Stabilisation
            fusion_tokens = torch.tanh(fusion_tokens) * alpha_train
            fusion_tokens = fusion_tokens.to(dtype=positive_embedding.dtype)

            # Prompt + fusion tokens
            conditioned_positive_embedding = torch.cat([positive_embedding, fusion_tokens], dim=1)

            # Passage dans le transformer de Stable Diffusion 3.5
            # Le modèle prédit la direction de débruitage à partir du latent bruité et du prompt enrichi
            model_pred = pipe.transformer(
                hidden_states=noisy_latents,
                timestep=timesteps,
                encoder_hidden_states=conditioned_positive_embedding,
                pooled_projections=positive_pooled_embedding,
                return_dict=False,
            )[0]

            # Calcul de la loss
            # Comparaison de la prédiction du modèle avec la cible attendue
            loss = F.mse_loss(model_pred.float(), target.float())
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss non finie à la step {global_steps}: {loss.item()}")

            running_count += 1
            running_loss += loss.item()

            # Backprop - mise à jour de l'adapter uniquement
            optimizer.zero_grad() # Effacer les gradients précédents
            loss.backward() # Calculer les nouveaux gradients
            grad_norm = torch.nn.utils.clip_grad_norm_(adapter.parameters(), 0.5) # Empêcher les gradients de devenir trop grands - mécanisme de stabilisation de l’entraînement

            if not torch.isfinite(grad_norm):
                raise RuntimeError(f"Gradient non fini a la step {global_steps}: {grad_norm}")

            optimizer.step() # Modifier les poids de l'Adapter
            if any(not torch.isfinite(p).all() for p in adapter.parameters()):
                raise RuntimeError(f"Poids adapter non finis apres la step {global_steps}")

            # Mise à jour de la barre de progression
            global_steps += 1

            progress_bar.update(1)
            progress_bar.set_postfix({ "loss": loss.item(), "step": global_steps })

            # Enregistrement des métriques dans WandB
            if global_steps % LOG_WANDB_STEPS == 0:
              avg_loss = running_loss / running_count

              print(
                  f"{global_steps:>6} "
                  f"{loss.item():>15.6f} "
                  f"{avg_loss:>15.6f} "
                  f"{float(grad_norm):>12.4f}"
              )

              wandb_log(avg_loss, loss, optimizer, alpha_train, grad_norm, global_steps, item_name, adapter, embedding)

              running_count = 0
              running_loss = 0

    progress_bar.close()

In [ ]:
# Coefficient qui limite l'influence des tokens de fusion au début de l'entraînement
alpha_train = 0.1

fusion_list = [
    {
        "fusion": cnn_embeddings,
        "name": "cnn_embeddings",
        "checkpoint_file": f"cnn_embeddings_adapter_{STEPS}_steps.pt",
        "name_wandb": "adapter-cnn-8-tokens-t4"
    },
    {
        "fusion": gnn_embeddings,
        "name": "gnn_embeddings",
        "checkpoint_file": f"gnn_embeddings_adapter_{STEPS}_steps.pt",
        "name_wandb": "adapter-gnn-8-tokens-t4"
    },
    {
        "fusion": urban_fusion,
        "name": "urban_fusion",
        "checkpoint_file": f"urban_fusion_adapter_{STEPS}_steps.pt",
        "name_wandb": "adapter-urban-fusion-8-tokens-t4"
    }
]

In [ ]:
idx = 0

if SELECTED_FUSION == "gnn_embeddings": idx = 1
elif SELECTED_FUSION == "urban_fusion": idx = 2

item = fusion_list[idx]

### **Training Adapter**

In [ ]:
adapter = LightweightUrbanFusionAdapter(input_dim=item["fusion"].shape[-1]).to(DEVICE)

save_adapter(adapter, item["fusion"], item["name"])

dataset = UrbanFusionDataset(
    urban_data=urban_data,
    fusion=item["fusion"],
    coco_root=COCO_ROOT,
    image_transform=image_transform
)

data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

run_wandb(adapter, item["name_wandb"])

# Optimizer pour l’adapter
optimizer = torch.optim.AdamW(
    adapter.parameters(),
    lr=LEARNING_RATE, # Vitesse d'apprentissage
    weight_decay=1e-2 # Régularisation pour limiter le surapprentissage
)

gc.collect()
torch.cuda.empty_cache()

# Passage de l'adapter en mode entraînement
adapter.train()
train_adapter(data_loader, STEPS, adapter, alpha_train, optimizer, item["name"])

save_training_checkpoint(adapter, optimizer, STEPS, item["fusion"], alpha_train, item["checkpoint_file"])
wandb.finish()